# InvokeAI on Google Colab — persistent setup (Google Drive)

Runs the latest InvokeAI release (web UI) on a Colab GPU, with all state stored on
Google Drive so nothing is lost when the runtime stops and you start it again.

**What persists on Drive** (`INVOKEAI_ROOT`, default `My Drive/invokeai`): downloaded
models, generated images (`outputs`), the database (boards, gallery, workflows) and
`invokeai.yaml`. The InvokeAI package itself is reinstalled into Colab's fast local disk
each fresh runtime (a few minutes) — only the data lives on Drive.

**Downloading models**: do it inside the running app — **Model Manager** lets you paste a
HuggingFace / CivitAI / direct URL and InvokeAI downloads it straight into the Drive root
itself. For CivitAI, add your API key in the Model Manager settings first.

## GPU
Use the most powerful GPU Google offers: **A100 (40 GB)** — the only Colab GPU that fits
the large modern models (Flux.1/Flux.2, SD 3.5 Large, Qwen Image) without heavy
offloading. Requires Colab Pro / Pro+. Fallback order: A100 > L4 (24 GB) > T4. Set it in
`Runtime -> Change runtime type -> A100 GPU` before running.

## How to use
1. Select an A100 (or L4) GPU runtime.
2. Fill the fields in the **Start InvokeAI** cell and run it (authorize Google Drive when
   asked). It prints a public `trycloudflare.com` URL and the login — open it once the
   server is up.
3. To stop, kill the runtime in Colab: **Runtime → Disconnect and delete runtime**. To
   resume later, just run the cell again; your models and images are still on Drive.

In [ ]:
#@title  Start InvokeAI  { display-mode: "form" }
#@markdown Runs the latest **InvokeAI** on this Colab GPU with all state saved to
#@markdown **Google Drive**, behind a **password-protected** public link. Fill the fields
#@markdown below and run this cell — it prints a `trycloudflare.com` URL when ready.
#@markdown To stop, just kill the runtime in Colab (**Runtime → Disconnect and delete
#@markdown runtime**); your models and images stay on Drive.
#@markdown
#@markdown **Google Drive location** ("My Drive" = personal drive, or a Shared Drive name)
DRIVE_NAME = "My Drive"  #@param {type:"string"}
INVOKEAI_FOLDER = "invokeai"  #@param {type:"string"}
#@markdown **Public link login** (leave the password empty to disable the password)
AUTH_USER = "invoke"  #@param {type:"string"}
AUTH_PASSWORD = "invoke"  #@param {type:"string"}
#@markdown **Performance**: install xformers (memory-efficient attention; modest speedup)
USE_XFORMERS = True  #@param {type:"boolean"}

import os
import sys
import re
import time
import subprocess
import urllib.request
import urllib.error

# --- Environment check ---
print(f"Python: {sys.version.split()[0]}")
if not (3, 11) <= sys.version_info[:2] <= (3, 12):
    print("WARNING: InvokeAI needs Python 3.11-3.12; this runtime may fail to install.")
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip()
    print(f"GPU: {gpu}")
    if "A100" not in gpu:
        print("WARNING: A100 not detected. For large models pick 'A100 GPU' in "
              "Runtime -> Change runtime type (Colab Pro/Pro+).")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("ERROR: No NVIDIA GPU. Set Runtime -> Change runtime type -> A100 GPU, then rerun.")

# --- Mount Google Drive and pin INVOKEAI_ROOT there for persistence ---
from google.colab import drive
drive.mount("/content/drive")

# "My Drive" is your personal drive; any other value is treated as a Shared Drive name.
drive_name = DRIVE_NAME.strip()
if drive_name.lower().replace(" ", "") in ("mydrive", ""):
    drive_base = "/content/drive/MyDrive"
else:
    drive_base = f"/content/drive/Shareddrives/{drive_name}"
if not os.path.isdir(drive_base):
    raise FileNotFoundError(
        f"Drive path not found: {drive_base}. Set DRIVE_NAME to 'My Drive' for your "
        f"personal drive, or to the exact name of a Shared Drive you can access."
    )

os.environ["INVOKEAI_ROOT"] = os.path.join(drive_base, INVOKEAI_FOLDER)
os.environ["INVOKEAI_HOST"] = "127.0.0.1"
os.environ["INVOKEAI_PORT"] = "9090"
os.makedirs(os.environ["INVOKEAI_ROOT"], exist_ok=True)
print(f"INVOKEAI_ROOT = {os.environ['INVOKEAI_ROOT']}")

# --- Install the latest InvokeAI (skip if already present this session) ---
try:
    import invokeai.version
    print(f"InvokeAI already installed: {invokeai.version.__version__}")
except ImportError:
    if USE_XFORMERS:
        # The [xformers] extra pins a matching torch build, so let pip resolve torch itself
        # (no cu128 index here). xformers adds memory-efficient attention.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", "invokeai[xformers]"],
            check=True,
        )
    else:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", "invokeai[cuda]",
             "--extra-index-url", "https://download.pytorch.org/whl/cu128"],
            check=True,
        )

# --- Fix Colab's protobuf: diffusers needs >=5.26 ('runtime_version'); Colab ships older ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.26,<6"], check=True)

# --- Download cloudflared and caddy (idempotent) ---
if not os.path.exists("/usr/local/bin/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared",
    )
    os.chmod("/usr/local/bin/cloudflared", 0o755)
if AUTH_PASSWORD and not os.path.exists("/usr/local/bin/caddy"):
    urllib.request.urlretrieve(
        "https://caddyserver.com/api/download?os=linux&arch=amd64", "/usr/local/bin/caddy"
    )
    os.chmod("/usr/local/bin/caddy", 0o755)

port = os.environ["INVOKEAI_PORT"]
proxy_port = "9091"
server_log = "/content/invokeai.log"

# --- Launch InvokeAI in the background ---
subprocess.Popen(["invokeai-web"], stdout=open(server_log, "w"), stderr=subprocess.STDOUT)

# Wait until the server is listening (a response, even 404, means it is up).
print("Starting InvokeAI server (first launch can take a few minutes)...")
server_ready = False
for _ in range(120):
    time.sleep(5)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=5)
    except urllib.error.HTTPError:
        server_ready = True
        break
    except Exception:
        continue
    server_ready = True
    break

if not server_ready:
    print(f"ERROR: server did not start. Last lines of {server_log}:")
    print(subprocess.run(["tail", "-n", "30", server_log], capture_output=True, text=True).stdout)
else:
    print("InvokeAI server is up.")
    # Optional HTTP Basic Auth in front of InvokeAI via a Caddy reverse proxy.
    if AUTH_PASSWORD:
        pw_hash = subprocess.run(
            ["/usr/local/bin/caddy", "hash-password", "--plaintext", AUTH_PASSWORD],
            capture_output=True, text=True,
        ).stdout.strip()
        caddyfile = (
            f":{proxy_port} {{\n"
            f"\tbasic_auth {{\n"
            f"\t\t{AUTH_USER} {pw_hash}\n"
            f"\t}}\n"
            f"\treverse_proxy 127.0.0.1:{port}\n"
            f"}}\n"
        )
        with open("/content/Caddyfile", "w") as caddy_conf:
            caddy_conf.write(caddyfile)
        subprocess.Popen(
            ["/usr/local/bin/caddy", "run", "--config", "/content/Caddyfile", "--adapter", "caddyfile"],
            stdout=open("/content/caddy.log", "w"), stderr=subprocess.STDOUT,
        )
        # Wait for Caddy: a 401 means it is up and enforcing auth.
        for _ in range(15):
            time.sleep(1)
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{proxy_port}/", timeout=3)
            except urllib.error.HTTPError:
                break
            except Exception:
                continue
            break
        tunnel_target = proxy_port
    else:
        print("WARNING: AUTH_PASSWORD is empty - the public link will be UNPROTECTED.")
        tunnel_target = port

    # --- Expose it through a cloudflared quick tunnel (no signup) ---
    tunnel_log = "/content/cloudflared.log"
    subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{tunnel_target}"],
        stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT,
    )
    public_url = None
    for _ in range(30):
        time.sleep(2)
        with open(tunnel_log) as log_file:
            match = re.search(r"https://[-\w]+\.trycloudflare\.com", log_file.read())
        if match:
            public_url = match.group(0)
            break
    print("=" * 60)
    print(f"Open InvokeAI at: {public_url or 'tunnel not ready - see ' + tunnel_log}")
    if AUTH_PASSWORD:
        print(f"Login: {AUTH_USER} / {AUTH_PASSWORD}")
    print(f"Server log: {server_log}")
    print("=" * 60)

In [ ]:
## Notes

- **Persistence**: everything under `INVOKEAI_ROOT` (default `/content/drive/MyDrive/invokeai`)
  is on Google Drive — models, `outputs`, `databases/invokeai.db` and `invokeai.yaml` all
  survive a runtime stop. Next session just run the Start cell again.
- **Always use the same root**: keep `DRIVE_NAME = My Drive` and `INVOKEAI_FOLDER = invokeai`
  so it resolves to `/content/drive/MyDrive/invokeai` every time. Mixing this with other
  notebooks/paths (e.g. `MyDrive/InvokeAI`) splits the database from the files and causes
  missing-model / 404-thumbnail errors.
- **Logs**: the server runs in the background, so its output goes to `/content/invokeai.log`,
  not the Start cell. Use the **View server logs & paths** cell to read it and confirm where
  data is stored.
- **Stopping**: kill the runtime in Colab — **Runtime → Disconnect and delete runtime** (or
  Interrupt). Drive autosaves; to be safe, stop generating a few seconds before killing it
  so the database finishes writing.
- **Password**: set `AUTH_USER` / `AUTH_PASSWORD`. A Caddy reverse proxy enforces HTTP Basic
  Auth on `:9091` and cloudflared tunnels that port. Empty password = unprotected link.
- **xformers**: `USE_XFORMERS` installs `invokeai[xformers]` (memory-efficient attention).
  It pins its own torch build, so the cu128 index is skipped in that mode. Untick to use
  `invokeai[cuda]` with cu128 wheels instead.
- **protobuf**: Start upgrades protobuf to >=5.26 because Colab ships an older build that
  crashes diffusers (`cannot import name 'runtime_version'`), which otherwise leaves the
  server down and the tunnel at 502 Bad Gateway.
- **Downloading models**: use the in-app **Model Manager** — paste a HuggingFace / CivitAI
  / direct URL and InvokeAI downloads it into the Drive root itself.
- **Drive space**: large models (Flux, SD3.5 Large, Qwen ~40 GB) consume a lot of Drive
  quota — keep enough free space.
- **Tunnel URL** changes every launch (quick tunnels are ephemeral). For a stable URL,
  switch cloudflared for an authenticated ngrok / cloudflare named tunnel.
- **GPU**: A100 (40 GB) gives the best throughput and fits the largest models; L4/T4 work
  for SDXL/SD1.5 but may need offloading for Flux-class models.

## Notes

- **Persistence**: everything under `INVOKEAI_ROOT` (default `My Drive/invokeai`) is on
  Google Drive — models, `outputs`, `databases/invokeai.db` and `invokeai.yaml` all
  survive a runtime stop. Next session just run the Start cell again.
- **Stopping**: kill the runtime in Colab — **Runtime → Disconnect and delete runtime** (or
  Interrupt). Drive autosaves; to be safe, stop generating a few seconds before killing it
  so the database finishes writing.
- **Drive location**: set `DRIVE_NAME` (`My Drive` or a Shared Drive name) and
  `INVOKEAI_FOLDER` in the Start form to change where data is stored.
- **Password**: set `AUTH_USER` / `AUTH_PASSWORD`. A Caddy reverse proxy enforces HTTP Basic
  Auth on `:9091` and cloudflared tunnels that port. Empty password = unprotected link.
- **xformers**: `USE_XFORMERS` installs `invokeai[xformers]` (memory-efficient attention).
  It pins its own torch build, so the cu128 index is skipped in that mode. Untick to use
  `invokeai[cuda]` with cu128 wheels instead.
- **protobuf**: Start upgrades protobuf to >=5.26 because Colab ships an older build that
  crashes diffusers (`cannot import name 'runtime_version'`), which otherwise leaves the
  server down and the tunnel at 502 Bad Gateway.
- **Downloading models**: use the in-app **Model Manager** — paste a HuggingFace / CivitAI
  / direct URL and InvokeAI downloads it into the Drive root itself.
- **Drive space**: large models (Flux, SD3.5 Large, Qwen ~40 GB) consume a lot of Drive
  quota — keep enough free space.
- **Tunnel URL** changes every launch (quick tunnels are ephemeral). For a stable URL,
  switch cloudflared for an authenticated ngrok / cloudflare named tunnel.
- **GPU**: A100 (40 GB) gives the best throughput and fits the largest models; L4/T4 work
  for SDXL/SD1.5 but may need offloading for Flux-class models.